<a href="https://colab.research.google.com/github/kenzoyanome/Proyecto001_rappiplus/blob/main/Proyecto_rapplus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**  
El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:
- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:
1. Evaluar si podemos confiar en los datos (calidad de datos en Python)
2. Analizar si el negocio es rentable (revenue, costos y profit)  
3. Entender dónde se pierden los usuarios (funnel de conversión)  
4. Evaluar si los usuarios regresan (retención por cohortes)  
5. Validar si los cambios generan impacto (test estadístico)  
6. Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## Paso 1: Cargar y validar la calidad de los datos
---
### 1.1 Carga de datos y vista rápida
**Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**
- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.
---

In [ ]:
# importar librerias
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns

from scipy.stats                  import pointbiserialr, chi2_contingency, ttest_ind, levene #pointbiserialr (punto biserial), chi2_contingency (chi square), ttest_ind, ttest_1samp, mannwhitneyu, levene, shapiro
from statsmodels.stats.proportion import proportions_ztest
# OTRAS CONFIGURACIONES: EN GENERAL, EN BLOQUE DE IMPORTS, SUELE EXISTIR UN ESPACIO DONDE SE HACEN DEFINICIONES GENERALES
#                       COMO ELIMINAR CIERTOS "WARNINGS" MOLESTOS, QUITAR LÍMITES DE PANDAS SOBRE CUANTAS FILAS O COLUMNAS
#                       MUESTRA AL MOMENTO DE IMPIRMIR EL DATAFRAME, DEFINIR ESTILOS PRE-DEFINIDOS PARA GRÁFICOS, ETC.
import warnings                                                  # MANEJO DE WARNINGS - ADVERTENCIAS
warnings.filterwarnings('ignore', category=FutureWarning)        # IGNORAR WARNINGS MOLESTOS
pd.set_option('display.max_columns', None)                       # ELIMINA LIMITES DE PANDAS PARA MOSTRAR COLUMNAS
pd.set_option('display.max_rows', None)                          # ELIMINA LIMITES DE PANDAS PARA MOSTRAR FILAS
pd.set_option('display.max_colwidth', None)                      # AUTOAJUSTA ANCHO DE COLUMNAS
pd.set_option('display.float_format', lambda x: '%.3f' % x)      # PERMITE EVADIR EL MOSTRAR NÚMEROS CON NOTACIÓN CINETÍFICA
plt.rcParams['figure.dpi'] = 140                                 # NIVEL DE RESOLUCIÓN.

In [ ]:
# cargar archivos
orders    = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/Proyecto001_rappiplus/refs/heads/main/rappiplus_orders_raw.csv')
catalog   = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/Proyecto001_rappiplus/refs/heads/main/rappiplus_catalog.csv')
marketing = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/Proyecto001_rappiplus/refs/heads/main/rappiplus_marketing_spend.csv')

In [ ]:
# explorar datasets
display(orders.head())
display(catalog.head())
display(marketing.head())

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.000,332.690,0.000,665.370
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.000,176.860,5.000,171.860
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.000,102.990,10.000,195.990
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.000,257.870,15.000,242.870
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.000,336.280,0.000,336.280


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.680,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.120,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.210,Bowers LLC
3,Blender-XL-Red,Hogar,176.640,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.600,"Rivera, Carr and Finley"


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.250
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.340
2,2025-01-01,Mexico,social_Mexico,social,2045.010
3,2025-01-01,Colombia,organic_Colombia,organic,2597.210
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.400


In [ ]:
# explorar datasets
print("orders filas y columnas:   ", orders.shape)
print("catalog filas y columnas:  ", catalog.shape)
print("marketing filas y columnas:", marketing.shape)

orders filas y columnas:    (25100, 12)
catalog filas y columnas:   (7, 4)
marketing filas y columnas: (1620, 5)


In [ ]:
# explorar datasets
print(orders.info(), '\n', '_'*50)
print(catalog.info(), '\n', '_'*50)
print(marketing.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
None 
 __________________________________________________
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 colum

---

### Revisión y calidad de datos

**Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto
- Revisar variables numéricas (sin negativos o ceros inválidos)
- Verificar consistencia de montos
- Eliminar duplicados
- Revisar variables categóricas
---

- Validar y convertir fechas al formato correcto

In [ ]:
# corregir formatos de fecha
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], format='%Y-%m-%d', errors='coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], format='%Y-%m-%d', errors='coerce')

# -verificacion de fechas modificadas
print(orders.info(), '\n', '_'*50)
print(marketing.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB
None 
 _____________________________

- Revisar variables numéricas (sin negativos o ceros inválidos)

In [ ]:
orders.describe().round(4)
# no se muestran 0's invalidos
# 'cantidad'       : 4 negativos se pueden cambiar por absolutos
#                  : 10 outliers de 10000 y 20000 se pueden dividir /10000
#                  : 50 nulos a descartar?
#                  : ------------------------- mismas filas que: 'precio_unitario' y 'monto_descuento'
# 'precio_unitario': 50 nulos a descartar?
#                  : ------------------------- mismas filas que: 'cantidad' y 'monto_descuento'
# 'monto_descuento': 50 nulos a descartar?
#                  : ------------------------- mismas filas que: 'cantidad' y 'precio_unitario'
# 'monto_total'    : no presenta datos faltantes, sin embargo crearemos una nueva columna con la formula de 'cantidad' * 'precio_unitario' - 'monto_descuento'

,fecha_hora_pedido,cantidad,precio_unitario,monto_descuento,monto_total
count,25100,25050.000,25050.000,25050.000,25100.000
mean,2025-04-01 03:21:22.231075584,7.093,259.305,4.501,2072.680
min,2025-01-01 00:00:00,-2.000,20.030,0.000,-492.650
25%,2025-02-15 00:00:00,1.000,138.377,0.000,180.507
50%,2025-04-02 00:00:00,2.000,258.715,0.000,341.750
75%,2025-05-16 00:00:00,2.000,380.332,10.000,518.580
max,2025-06-30 00:00:00,20000.000,499.960,15.000,8840200.000
std,NaN,296.277,138.726,5.223,98949.950


In [ ]:
# cambiamos valores negativos por absolutos y confirmamos cambios
orders['cantidad'] = orders['cantidad'].abs()
orders.describe().round(4)

,fecha_hora_pedido,cantidad,precio_unitario,monto_descuento,monto_total
count,25100,25050.000,25050.000,25050.000,25100.000
mean,2025-04-01 03:21:22.231075584,7.093,259.305,4.501,2072.680
min,2025-01-01 00:00:00,1.000,20.030,0.000,-492.650
25%,2025-02-15 00:00:00,1.000,138.377,0.000,180.507
50%,2025-04-02 00:00:00,2.000,258.715,0.000,341.750
75%,2025-05-16 00:00:00,2.000,380.332,10.000,518.580
max,2025-06-30 00:00:00,20000.000,499.960,15.000,8840200.000
std,NaN,296.277,138.726,5.223,98949.950


In [ ]:
# revisamos outliers
Q1 = orders['cantidad'].quantile(0.25)
Q2 = orders['cantidad'].quantile(0.50)
Q3 = orders['cantidad'].quantile(0.75)
IQR = Q3 - Q1
lowerl = Q1 - 1.5 * IQR
upperl = Q3 + 1.5 * IQR
print(f"cantidad: lower limit: {lowerl}")
print(f"cantidad: upper limit: {upperl}")

# mostramos outliers
orders_outliers = orders[ (orders['cantidad'] < lowerl) | (orders['cantidad'] > upperl) ]
display(orders_outliers.head(3))

outliers_listado = orders_outliers.index
print(orders.loc[outliers_listado, 'cantidad'])

cantidad: lower limit: -0.5
cantidad: upper limit: 3.5


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.000,43.140,0.000,431400.000
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electronica,10000.000,280.550,0.000,2805500.000
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.000,490.350,0.000,4903500.000


3521   10000.000
3522   10000.000
3586   10000.000
3643   10000.000
3656   20000.000
3668   20000.000
3689   10000.000
3722   20000.000
3726   20000.000
3748   10000.000
Name: cantidad, dtype: float64


In [ ]:
# imputar outliers dividiendolos /10,000
orders.loc[outliers_listado, 'cantidad'] = orders.loc[outliers_listado, 'cantidad'] / 10000

# mostrar cambios
orders.describe().round(3)

,fecha_hora_pedido,cantidad,precio_unitario,monto_descuento,monto_total
count,25100,25050.000,25050.000,25050.000,25100.000
mean,2025-04-01 03:21:22.231075584,1.505,259.306,4.501,2072.680
min,2025-01-01 00:00:00,1.000,20.030,0.000,-492.650
25%,2025-02-15 00:00:00,1.000,138.378,0.000,180.508
50%,2025-04-02 00:00:00,2.000,258.715,0.000,341.750
75%,2025-05-16 00:00:00,2.000,380.332,10.000,518.580
max,2025-06-30 00:00:00,2.000,499.960,15.000,8840200.000
std,NaN,0.500,138.726,5.223,98949.950


- Verificar consistencia de montos

In [ ]:
# creamos una nueva columna 'monto_esperado' ('cantidad' * 'precio_unitario' - 'monto_descuento') y comparamos con 'monto_total'
orders['monto_esperado'] = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']

# mostramos resultados y comparativa
display(orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'monto_esperado']].head())
print(orders[['monto_esperado', 'monto_total']].describe())

,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado
0,2.000,332.690,0.000,665.370,665.380
1,1.000,176.860,5.000,171.860,171.860
2,2.000,102.990,10.000,195.990,195.980
3,1.000,257.870,15.000,242.870,242.870
4,1.000,336.280,0.000,336.280,336.280


       monto_esperado  monto_total
count       25050.000    25100.000
mean          385.807     2072.680
std           255.746    98949.950
min             5.240     -492.650
25%           180.343      180.507
50%           341.365      341.750
75%           517.655      518.580
max           999.880  8840200.000


In [ ]:
# analizamos nulos
print(orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'monto_esperado']].isna().sum())
print(orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'monto_esperado']].isna().sum() /25100 * 100)
print("_"*50)
columnas_nulos = ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_esperado']
for cols in columnas_nulos:
    nulos = orders[orders[cols].isnull()].index.tolist() # -para hacer una comparativa de las filas en cada uno
    print(f"filas con nulos en {cols}\n{nulos}")
# se observa que los nulos son en los mismos indices, 50 filas que podemos eliminar (representan un 0.199 % de los datos totales)

cantidad           50
precio_unitario    50
monto_descuento    50
monto_total         0
monto_esperado     50
dtype: int64
cantidad          0.199
precio_unitario   0.199
monto_descuento   0.199
monto_total       0.000
monto_esperado    0.199
dtype: float64
__________________________________________________
filas con nulos en cantidad
[74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123]
filas con nulos en precio_unitario
[74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123]
filas con nulos en monto_descuento
[74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107

In [ ]:
# eliminamos nulos de 'cantidad'
orders = orders.dropna(subset=['cantidad']).reset_index(drop=True)
orders.info()
# se eliminan a la vez los nulos de 'precio_unitario', 'monto_descuento' y 'monto_esperado'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25050 entries, 0 to 25049
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25050 non-null  object        
 1   id_usuario          25050 non-null  object        
 2   fecha_hora_pedido   25050 non-null  datetime64[ns]
 3   pais                24750 non-null  object        
 4   dispositivo         25030 non-null  object        
 5   fuente_referencia   25020 non-null  object        
 6   nombre_producto     25020 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25050 non-null  float64       
 12  monto_esperado      25050 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(7)
me

- Eliminar duplicados

In [ ]:
# revision de duplicados
print('cantidad de filas duplicadas:', orders.duplicated().sum())
print("_"*50)
orders[orders.duplicated(keep=False)].sort_values('id_usuario').head()
# observamos 100 filas duplicadas que pueden ser eliminadas

cantidad de filas duplicadas: 100
__________________________________________________


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado
24999,order_3167,user_1040,2025-01-08,Mexico,desktop,organic,Vacuum-Pro-Black,Hogar,1.000,202.600,0.000,202.600,202.600
3117,order_3167,user_1040,2025-01-08,Mexico,desktop,organic,Vacuum-Pro-Black,Hogar,1.000,202.600,0.000,202.600,202.600
6743,order_6793,user_1115,2025-01-22,Argentina,desktop,organic,Phone-Pro-128GB,Electronica,1.000,154.890,5.000,149.890,149.890
24972,order_6793,user_1115,2025-01-22,Argentina,desktop,organic,Phone-Pro-128GB,Electronica,1.000,154.890,5.000,149.890,149.890
20403,order_20453,user_1237,2025-04-16,Colombia,desktop,organic,Phone-Pro-128GB,Electronica,1.000,461.930,0.000,461.930,461.930


In [ ]:
# eliminamos duplicados y solo nos quedamos con la primera fila de cada uno
orders = orders.drop_duplicates(keep='first').reset_index(drop=True)

# validamos que no queden duplicados
print('cantidad de filas duplicadas:', orders.duplicated().sum())
print(orders.info())

cantidad de filas duplicadas: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24950 entries, 0 to 24949
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24950 non-null  object        
 1   id_usuario          24950 non-null  object        
 2   fecha_hora_pedido   24950 non-null  datetime64[ns]
 3   pais                24650 non-null  object        
 4   dispositivo         24930 non-null  object        
 5   fuente_referencia   24920 non-null  object        
 6   nombre_producto     24920 non-null  object        
 7   categoria_producto  24920 non-null  object        
 8   cantidad            24950 non-null  float64       
 9   precio_unitario     24950 non-null  float64       
 10  monto_descuento     24950 non-null  float64       
 11  monto_total         24950 non-null  float64       
 12  monto_esperado      24950 non-null  float64       
dtypes: datetime64[

- Revisar variables categóricas

In [ ]:
# -definir columnas categoricas
categoricas_orders    = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']
categoricas_catalog   = ['nombre_producto', 'categoria_producto', 'proveedor']
categoricas_marketing = ['pais', 'id_campaña', 'canal']

# -funcion para corregir valores unicos y conteo de nulos
def rev_categorica (data, categoricas):
    for col in categoricas:
        nulos = data[col].isna().sum()
        print(f"Revisión de columna: {col}  |  Nulos: {nulos} ({nulos/len(data)*100:.4f}%)")
        display(data[col].value_counts(dropna = False))
        print("_"*50)

# -aplicacion de funcion rev_categorica
rev_categorica(data = orders,    categoricas = categoricas_orders)
rev_categorica(data = catalog,   categoricas = categoricas_catalog)
rev_categorica(data = marketing, categoricas = categoricas_marketing)

Revisión de columna: pais  |  Nulos: 300 (1.2024%)


,count
pais,
Colombia,7467
Mexico,7465
Argentina,7239
mexico,862
colombia,822
argentina,795
NaN,300


__________________________________________________
Revisión de columna: dispositivo  |  Nulos: 20 (0.0802%)


,count
dispositivo,
desktop,12685
mobile,12245
NaN,20


__________________________________________________
Revisión de columna: fuente_referencia  |  Nulos: 30 (0.1202%)


...

__________________________________________________
Revisión de columna: id_campaña  |  Nulos: 0 (0.0000%)


,count
id_campaña,
organic_Mexico,180
paid_search_Mexico,180
social_Mexico,180
organic_Colombia,180
paid_search_Colombia,180
social_Colombia,180
organic_Argentina,180
paid_search_Argentina,180
social_Argentina,180


__________________________________________________
Revisión de columna: canal  |  Nulos: 101 (6.2346%)


,count
canal,
paid_search,507
organic,506
social,506
NaN,101


__________________________________________________


In [ ]:
# -quitar espacios y dejar formato title en columnas categoricas
def strip_title (dato, categoricas):
    for col in categoricas:
        dato[col] = dato[col].str.strip().str.title()

# -aplicacion de funcion strip_title
strip_title(dato = orders,    categoricas = categoricas_orders)
strip_title(dato = catalog,   categoricas = categoricas_catalog)
strip_title(dato = marketing, categoricas = categoricas_marketing)

In [ ]:
# imputar o descartar valores en categoricas
print(orders.isna().sum(), '\n', '_'*50)
# 'pais'              : 300 nulos, la mayoria pueden imputarse en base al 'id_usuario'
#                     : los restantes (19) podrian eliminarse para no afectar los totales
# 'dispositivo'       : 20 nulos, pudieran imputarse, solo comprenden el 0.08% de los datos
# 'fuente_referencia' : 30 nulos a descartar, solo comprenden el 0.12% de los datos
#                     : ------------------------- mismas filas que: 'nombre_producto' y 'categoria_producto'
# 'nombre_producto'   : 30 nulos a descartar?, solo comprenden el 0.12% de los datos
#                     : ------------------------- mismas filas que: 'fuente_referencia' y 'categoria_producto'
# 'categoria_producto': 30 nulos a descartar?, solo comprenden el 0.12% de los datos
#                     : ------------------------- mismas filas que: 'fuente_referencia' y 'nombre_producto'
#                     : aun cuando en el .cvs aparecen 80 datos vacios, 50/80 pueden imputarse en base a 'nombre_producto'
print(catalog.isna().sum(), '\n', '_'*50)
#                     : no hay valores faltantes
print(marketing.isna().sum())
# 'canal'             : 101 nulos se puede imputar en base a 'id_campaña'

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
monto_esperado          0
dtype: int64 
 __________________________________________________
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64 
 __________________________________________________
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64


In [ ]:
# imputamos 'pais' en base a 'id_usuario'
# creamos un mapa usuario - país (tomando el primer valor no nulo)
mapa_pais = orders.dropna(subset=['pais']).groupby('id_usuario')['pais'].first()

# imputamos usando el mapa
orders['pais'] = orders['pais'].fillna(orders['id_usuario'].map(mapa_pais))

# confirmamos cuantos siguen nulos despues de imputar
print("paises que no pudieron imputarse:",  orders['pais'].isnull().sum())

# que usuarios no se pudieron imputar
sin_pais = orders[orders['pais'].isnull()][['id_pedido', 'id_usuario']]
print(sin_pais)

paises que no pudieron imputarse: 19
     id_pedido id_usuario
81   order_131  user_2914
122  order_172  user_1918
123  order_173  user_3776
138  order_188  user_7144
139  order_189   user_722
162  order_212  user_7893
174  order_224  user_4192
178  order_228  user_4917
225  order_275  user_3523
231  order_281  user_7788
234  order_284  user_4927
255  order_305  user_2527
268  order_318  user_6847
284  order_334  user_3525
302  order_352  user_5364
306  order_356  user_2386
320  order_370  user_4917
325  order_375  user_4992
342  order_392  user_2931


In [ ]:
# las filas restantes sin dato en 'pais' (19 filas = 0.076%) las eliminaremos
orders = orders.dropna(subset=['pais'])
print(orders.isna().sum())

id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo           20
fuente_referencia     30
nombre_producto       30
categoria_producto    30
cantidad               0
precio_unitario        0
monto_descuento        0
monto_total            0
monto_esperado         0
dtype: int64


In [ ]:
# imputamos 'categoria_producto' en base a 'nombre_producto'
# creamos un mapa categoria - nombre (tomando el primer valor no nulo)
mapa_categoria = orders.dropna(subset=['categoria_producto']).groupby('nombre_producto')['categoria_producto'].first()

# imputamos usando el mapa
orders['categoria_producto'] = orders['categoria_producto'].fillna(orders['nombre_producto'].map(mapa_categoria))

# confirmamos cuantos siguen nulos despues de imputar
print("categorias que no pudieron imputarse", orders['categoria_producto'].isnull().sum())

# que usuarios no se pudieron imputar
sin_categoria = orders[orders['categoria_producto'].isnull()][['id_pedido', 'nombre_producto']]
print(sin_categoria)

categorias que no pudieron imputarse 30
   id_pedido nombre_producto
44  order_44             NaN
45  order_45             NaN
46  order_46             NaN
47  order_47             NaN
48  order_48             NaN
49  order_49             NaN
50  order_50             NaN
51  order_51             NaN
52  order_52             NaN
53  order_53             NaN
54  order_54             NaN
55  order_55             NaN
56  order_56             NaN
57  order_57             NaN
58  order_58             NaN
59  order_59             NaN
60  order_60             NaN
61  order_61             NaN
62  order_62             NaN
63  order_63             NaN
64  order_64             NaN
65  order_65             NaN
66  order_66             NaN
67  order_67             NaN
68  order_68             NaN
69  order_69             NaN
70  order_70             NaN
71  order_71             NaN
72  order_72             NaN
73  order_73             NaN


In [ ]:
# eliminamos nulos en 'fuente_referencia' que eliminaria las mismas filas que: 'nombre_producto' y 'categoria_producto'
orders = orders.dropna(subset=['fuente_referencia'])

# eliminamos nulos de 'dispositivo'
orders = orders.dropna(subset=['dispositivo'])

# validamos cambios
print(orders.isna().sum())

id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
monto_esperado        0
dtype: int64


In [ ]:
# imputamos 'canal' en base a 'id_campaña' (sin el nombre del pais)
canal_nulo = marketing['canal'].isnull()
marketing.loc[canal_nulo, 'canal'] = marketing.loc[canal_nulo, 'id_campaña'].str.rsplit('_', n=1).str[0]

# verificamos cambios
print(marketing.isna().sum())

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64


In [ ]:
# -confirmamos que no queden nulos
print(orders.isna().sum(), '\n', '_'*50)
print(catalog.isna().sum(), '\n', '_'*50)
print(marketing.isna().sum())

id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
monto_esperado        0
dtype: int64 
 __________________________________________________
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64 
 __________________________________________________
fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64


---
**Exportación de datasets limpios**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
from google.colab import files

orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

files.download('orders_clean.csv')
files.download('catalog_clean.csv')
files.download('marketing_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Paso 2: Analizar si el negocio es rentable

---
### 2.1 Cálculo de KPIs principales

**Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

---

In [ ]:
# Cuál es el ingreso total (revenue)?
revenue = orders['monto_esperado'].sum()

# Cuál es el costo total?
#      unimos orders y catalog
orders_catalog = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
#      verificamos que no queden valores nulos
print("Nulos en costo unitario=", orders_catalog['costo_unitario'].isnull().sum())
costo_total = (orders_catalog['cantidad'] * orders_catalog['costo_unitario']).sum()

# Cuánto se ha invertido en marketing?
inversion_marketing = marketing['gasto'].sum()

# El negocio es rentable? (calcular profit)
profit = revenue - costo_total - inversion_marketing
margen_total = (profit / revenue) * 100

print('-----Resumen Financiero-----')
print(f"revenue total:                $ {revenue:.2f}")
print(f"costo total:                  $ {costo_total:.2f}")
print(f"inversion total de marketing: $ {inversion_marketing:.2f}")
print(f"profit:                       $ {profit:.2f}")
print(f"margen total:                   {margen_total:.2f}%")

Nulos en costo unitario= 0
-----Resumen Financiero-----
revenue total:                $ 9602232.34
costo total:                  $ 3828316.01
inversion total de marketing: $ 2871843.53
profit:                       $ 2902072.80
margen total:                   30.22%


In [ ]:
# Cuál es el ticket promedio por orden?
ticket_promedio = orders.groupby('id_pedido')['monto_esperado'].sum().mean()

# Cuál es la cantidad promedio de productos por orden?
cant_promedio_por_orden = orders.groupby('id_pedido')['cantidad'].sum().mean()

# ¿Cuál es el producto más vendido?
producto_mas_vendido = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)

# ¿Cuánto se ha gastado en marketing por canal?
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print(f"ticket promedio por orden:    $ {ticket_promedio:.2f}")
print(f"cantidad promedio por orden:    {cant_promedio_por_orden:.2f}")
print('_'*50)
print(f"top 3 productos mas vendidos: \n{producto_mas_vendido.head(3)}")
print('_'*50)
print(f"gasto de marketing por canal:")
print(gasto_por_canal.apply (lambda x: f"$ {x: ,.2f}"))

ticket promedio por orden:    $ 385.93
cantidad promedio por orden:    1.51
__________________________________________________
top 3 productos mas vendidos: 
nombre_producto
Vacuum-Pro-Black   6274.000
Blender-Xl-Red     6268.000
Jacket-Winter-M    6251.000
Name: cantidad, dtype: float64
__________________________________________________
gasto de marketing por canal:
canal
Social         $  976,818.37
Organic        $  972,650.96
Paid_Search    $  922,374.20
Name: gasto, dtype: object


___

## Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.

**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?
- Se calcula el número de usuarios únicos por `nombre_evento`
- Se ordenan los eventos según el flujo del usuario

---

**Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios
- Cuál es la tasa de conversión final?
---

In [ ]:
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
print(events.shape)
events.head()

(120000, 8)


,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# -Cuántos usuarios llegan a cada etapa del funnel?
# -Se calcula el número de usuarios únicos por `nombre_evento`
# -Se ordenan los eventos según el flujo del usuario
# ======================

query_totals = '''

SELECT nombre_evento, COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
WHERE nombre_evento IN ('first_visit', 'select_item', 'add_to_cart', 'begin_checkout', 'add_payment_info', 'purchase')
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
        WHEN 'first_visit'       THEN 1
        WHEN 'select_item'       THEN 2
        WHEN 'add_to_cart'       THEN 3
        WHEN 'begin_checkout'    THEN 4
        WHEN 'add_payment_info'  THEN 5
        WHEN 'purchase'          THEN 6
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

# -se observa un comportamiento anormal entre 'select_item' y 'add_to_cart'. Posible problema en el tracking de datos

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# -Se calcula la tasa de conversión entre cada paso del funnel
# -Se identifica en qué etapa se pierde la mayor cantidad de usuarios
# -Cuál es la tasa de conversión final?
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit'       THEN 1
            WHEN 'select_item'       THEN 2
            WHEN 'add_to_cart'       THEN 3
            WHEN 'begin_checkout'    THEN 4
            WHEN 'add_payment_info'  THEN 5
            WHEN 'purchase'          THEN 6
        END AS ordenado
    FROM events
    WHERE nombre_evento IN ('first_visit', 'select_item', 'add_to_cart', 'begin_checkout', 'add_payment_info', 'purchase')
    GROUP BY nombre_evento
)
SELECT nombre_evento, usuarios_unicos,
-- Usuarios perdidos vs etapa anterior
    (usuarios_unicos - LAG(usuarios_unicos) OVER (ORDER BY ordenado)) * -1  AS usuarios_perdidos_vs_etapa_previa,
-- Conversión respecto a etapa anterior
    CONCAT(ROUND(usuarios_unicos * 100.0
    / LAG(usuarios_unicos) OVER (ORDER BY ordenado), 2), '%%')              AS conversion_repecto_a_etapa_previa,
-- Conversión respecto al inicio (first_visit)
    CONCAT(ROUND(usuarios_unicos * 100.0
    / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY ordenado), 2), '%%')      AS conversion_total
FROM funnel
ORDER BY ordenado
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

# -'select_item' y 'add_to_cart' muestran un aumento de usuarios (-52) debido a posible error de tracking

,nombre_evento,usuarios_unicos,usuarios_perdidos_vs_etapa_previa,conversion_repecto_a_etapa_previa,conversion_total
0,first_visit,7796,NaN,%,100.00%
1,select_item,7582,214.000,97.26%,97.26%
2,add_to_cart,7634,-52.000,100.69%,97.92%
3,begin_checkout,7208,426.000,94.42%,92.46%
4,add_payment_info,6250,958.000,86.71%,80.17%
5,purchase,6240,10.000,99.84%,80.04%


___

## Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**
- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.
2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1
   - `retenido_w2`: usuarios activos en la semana 2
   - `retenido_w3`: usuarios activos en la semana 3
3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

---

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT id_usuario,
        fecha_registro,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte_mensual
    FROM users
    GROUP BY id_usuario, fecha_registro
),
joined AS(
    SELECT
    c.id_usuario,
    c.fecha_registro,
    c.cohorte_mensual,
    ua.dias_despues_registro,
    ua.activo
    FROM cohortes AS c
    LEFT JOIN user_activity AS ua
        ON c.id_usuario = ua.id_usuario
),
retencion_semanal AS (
    SELECT
    cohorte_mensual,
    COUNT(DISTINCT id_usuario) AS clientes_iniciales,
    COUNT(DISTINCT CASE WHEN dias_despues_registro BETWEEN 1 AND 7   AND activo = 1 THEN id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN dias_despues_registro BETWEEN 8 AND 14  AND activo = 1 THEN id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN dias_despues_registro BETWEEN 15 AND 21 AND activo = 1 THEN id_usuario END) AS retenido_w3,
    COUNT(DISTINCT CASE WHEN dias_despues_registro BETWEEN 22 AND 28 AND activo = 1 THEN id_usuario END) AS retenido_w4
    FROM joined
    GROUP BY cohorte_mensual
)
SELECT
    TO_CHAR(cohorte_mensual, 'YYYY-MM') AS cohorte,
    clientes_iniciales,
    ROUND((retenido_w1::numeric)*100 / clientes_iniciales, 2) AS semana_1,
    ROUND((retenido_w2::numeric)*100 / clientes_iniciales, 2) AS semana_2,
    ROUND((retenido_w3::numeric)*100 / clientes_iniciales, 2) AS semana_3,
    ROUND((retenido_w4::numeric)*100 / clientes_iniciales, 2) AS semana_4
FROM retencion_semanal
ORDER BY cohorte ASC, clientes_iniciales;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final.tail(8)

,cohorte,clientes_iniciales,semana_1,semana_2,semana_3,semana_4
0,2025-01,1627,42.840,41.060,40.320,41.240
1,2025-02,1444,42.310,42.170,43.980,39.820
2,2025-03,1636,41.380,43.090,42.180,41.140
3,2025-04,1606,42.340,43.400,41.280,40.600
4,2025-05,1687,41.200,40.070,41.850,40.250


___

## Paso 5: Validar si los cambios generan impacto (test estadístico)

**Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---
1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.  
2. **Plantear la hipótesis estadística**
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**

---

Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay diferencia. Las proporciones de conversion son iguales entre ambos grupos. Cualquier diferencia se debe al azar del muestreo.
   - **H₁ (Hipótesis alternativa):** Existe diferencia, La tasa de conversion es diferente entre ambos grupos. Hay un factor que influye en la decision de conversion.
   
**Test estadístico:** Z-test para comparar proporciones.  
**Nivel de significancia alpha:** 0.05

In [ ]:
ui = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/Proyecto001_rappiplus/refs/heads/main/experiment_checkout_ui.csv')
print(ui.info())
display(ui.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB
None


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.410,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.030,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.210,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.450,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.960,2025-01-12


In [ ]:
# tenemos una muestra de 10000 usuarios

# conversiones exitosas y totales
conversiones = ui.groupby('variante')['convirtio'].sum()
totales = ui.groupby('variante')['convirtio'].count()

# convertimos los resultados a listado
exitos = [conversiones['tratamiento'], conversiones['control']]
observaciones = [totales['tratamiento'], totales['control']]

#aplicamos z-test y visualizamos resultados
z_stat, p_value = proportions_ztest(exitos , observaciones)
print(f"Estadístico z: {z_stat:.4f}")
print(f"Valor p: {p_value:.4f}")

# Interpretar resultados
alpha = 0.05
if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia entre el grupo de control y el de tratamiento.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia entre el grupo de control y el de tratamiento.")

Estadístico z: 0.8133
Valor p: 0.4161
No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia entre el grupo de control y el de tratamiento.


___

## Paso 6: Comunicar los resultados (Dashboard en BI)

**Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---
1. Preparación de los datos
- Cargar los CSVs en Power BI o Tableau.
- Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---
2. Dashboard 1: Overview Ejecutivo  
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---
3. Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

Se incluyen archivos de Power BI y Tableau